# DK1 Electricity Price Forecasting — Calendar Model (Task 2)

**Econometric Game 2026 — Long-Horizon Forecasting**

Predict the weekly average of hourly spot prices for every week of 2024:
52 weeks × 24 hours = **1,248 forecasts**.

**Calendar OLS model:**

$$p_{t,k} = \sum_{k} \alpha_k \cdot \mathbb{1}[\text{hour}=k] + \beta_1 \cdot \text{weekend} + \beta_2 \cdot \text{holiday} + \sum_{m} \gamma_m \cdot \mathbb{1}[\text{month}=m] + \delta \cdot X_t + \varepsilon$$

Exogenous variables for 2024 are predicted using training-set conditional means
by (hour, month, is_weekend, is_holiday), with fallback to (hour, month, is_weekend).

---
## 0 · Imports & Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid', font_scale=1.0)
plt.rcParams['figure.dpi'] = 120

DATA = Path('..') / 'data'

# ── Configuration ─────────────────────────────────────────────────────────────
TRAIN_END = '2023-12-31'
TEST_YEAR = 2024

# Exogenous variables: chosen because they lack trend and can be predicted
# by simple conditional averages of historical values.
EXOG_HOURLY = [
    'wind_day_ahead_mw',
    'day_ahead_load_forecast_mw',
    'net_import_mw',
]
EXOG_DAILY = [
    'temperature',
    'precipitation',
    'sunshine',
    'wind_speed',
]
EXOG_COLS = EXOG_HOURLY + EXOG_DAILY

print(f'Exogenous variables ({len(EXOG_COLS)}): {EXOG_COLS}')

In [ ]:
# ── Evaluation helpers ─────────────────────────────────────────────────────────
def mae(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred)

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def mape(y_true, y_pred):
    mask = np.abs(y_true) > 1.0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(name, y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return pd.Series({
        'MAE':  round(mae(y_true, y_pred), 3),
        'RMSE': round(rmse(y_true, y_pred), 3),
        'MAPE': round(mape(y_true, y_pred), 3),
    }, name=name)

---
## 1 · Load & Prepare Data

In [ ]:
raw = pd.read_csv(
    DATA / 'hourly_features.csv',
    parse_dates=['ts_utc'],
    index_col='ts_utc',
)
raw.index = raw.index.tz_localize('UTC')

# Forward-fill daily variables (NaN on weekends/holidays for commodities)
for col in ['gas_price', 'eua_price', 'coal_price', 'oil_price',
            'temperature', 'precipitation', 'sunshine', 'wind_speed']:
    if col in raw.columns:
        raw[col] = raw[col].ffill()

# Calendar features
raw['date'] = raw.index.normalize().tz_localize(None)
raw['hour'] = raw.index.hour
raw['month'] = raw.index.month

print(f'Raw data: {raw.shape[0]:,} hourly obs, {raw.index.min().date()} to {raw.index.max().date()}')
print(f'Price NaN: {raw["day_ahead_price_eur_mwh"].isna().sum()}')

---
## 2 · Build Training Dataset

In [ ]:
# ── Filter to training period ─────────────────────────────────────────────────
train = raw.loc[:TRAIN_END].copy()
print(f'Training: {len(train):,} rows ({train.index.min().date()} to {train.index.max().date()})')

# ── Dependent variable ────────────────────────────────────────────────────────
y_train = train['day_ahead_price_eur_mwh'].copy()

# ── Dummy variables ───────────────────────────────────────────────────────────
# 23 hour dummies (drop hour 0 as reference)
hour_dummies = pd.get_dummies(train['hour'], prefix='hour', drop_first=True, dtype=float)
hour_dummies.index = train.index

# 11 month dummies (drop January as reference)
month_dummies = pd.get_dummies(train['month'], prefix='month', drop_first=True, dtype=float)
month_dummies.index = train.index

# Weekend and holiday
calendar_dummies = train[['is_weekend', 'is_holiday']].astype(float)

# ── Exogenous variables ───────────────────────────────────────────────────────
exog_train = train[EXOG_COLS].copy()

# ── Assemble feature matrix ───────────────────────────────────────────────────
X_train = pd.concat([hour_dummies, calendar_dummies, month_dummies, exog_train], axis=1)

# Drop rows with NaN
valid = y_train.notna() & X_train.notna().all(axis=1)
X_train = X_train[valid]
y_train = y_train[valid]

FEATURE_COLS = list(X_train.columns)
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS[:5]} ... {FEATURE_COLS[-5:]}')
print(f'Training observations: {len(X_train):,} (after dropping NaN)')

---
## 3 · Fit Calendar OLS

In [ ]:
# ── Fit OLS ───────────────────────────────────────────────────────────────────
ols = LinearRegression()
ols.fit(X_train, y_train)

y_train_pred = ols.predict(X_train)
r2 = ols.score(X_train, y_train)

print(f'R² (in-sample): {r2:.4f}')
print(f'Intercept:      {ols.intercept_:.3f}')
print()

# ── Coefficient table ─────────────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'coefficient': ols.coef_,
}, index=FEATURE_COLS)

print('=== Calendar Dummies & Exogenous Coefficients ===')
non_hour = [c for c in FEATURE_COLS if not c.startswith('hour_')]
print(coef_df.loc[non_hour].round(4))
print()
print('=== Hour Dummy Coefficients (relative to hour 0) ===')
hour_cols = [c for c in FEATURE_COLS if c.startswith('hour_')]
print(coef_df.loc[hour_cols].round(3))

---
## 4 · Predict Exogenous Variables for 2024

In [ ]:
# ── Compute conditional means from training data ──────────────────────────────
# Primary key: (hour, month, is_weekend, is_holiday)
group_cols_primary = ['hour', 'month', 'is_weekend', 'is_holiday']
cond_means_primary = train.groupby(group_cols_primary)[EXOG_COLS].mean()

# Fallback key: (hour, month, is_weekend)
group_cols_fallback = ['hour', 'month', 'is_weekend']
cond_means_fallback = train.groupby(group_cols_fallback)[EXOG_COLS].mean()

print(f'Primary conditional mean cells: {len(cond_means_primary)}')
print(f'Fallback conditional mean cells: {len(cond_means_fallback)}')

# ── Build 2024 target hours ───────────────────────────────────────────────────
test_raw = raw.loc['2024'].copy()
test_hours = test_raw[['date', 'hour', 'month', 'is_weekend', 'is_holiday']].copy()

print(f'\n2024 hours: {len(test_hours):,}')
print(f'2024 holidays: {test_hours["is_holiday"].sum():.0f} hours ({test_hours["is_holiday"].sum()/24:.0f} days)')

# ── Look up predicted exogenous values ────────────────────────────────────────
exog_predicted = pd.DataFrame(index=test_hours.index, columns=EXOG_COLS, dtype=float)

n_fallback = 0
for idx, row in test_hours.iterrows():
    key_primary = (row['hour'], row['month'], row['is_weekend'], row['is_holiday'])
    if key_primary in cond_means_primary.index:
        exog_predicted.loc[idx] = cond_means_primary.loc[key_primary]
    else:
        key_fallback = (row['hour'], row['month'], row['is_weekend'])
        exog_predicted.loc[idx] = cond_means_fallback.loc[key_fallback]
        n_fallback += 1

print(f'\nFallback used: {n_fallback} / {len(test_hours)} hours')
print(f'Any NaN in predicted exog: {exog_predicted.isna().any().any()}')
print(f'\nPredicted exog summary:')
exog_predicted.describe().round(2)

---
## 5 · Generate 2024 Hourly Forecasts

In [ ]:
# ── Build feature matrix for all 2024 hours ──────────────────────────────────
# Hour dummies
hour_dummies_test = pd.get_dummies(test_hours['hour'], prefix='hour', drop_first=True, dtype=float)
hour_dummies_test.index = test_hours.index

# Month dummies
month_dummies_test = pd.get_dummies(test_hours['month'], prefix='month', drop_first=True, dtype=float)
month_dummies_test.index = test_hours.index

# Weekend and holiday
calendar_test = test_hours[['is_weekend', 'is_holiday']].astype(float)

# Assemble
X_test = pd.concat([hour_dummies_test, calendar_test, month_dummies_test, exog_predicted], axis=1)

# Ensure column order matches training
X_test = X_test.reindex(columns=FEATURE_COLS, fill_value=0.0)

# ── Predict ───────────────────────────────────────────────────────────────────
hourly_forecast = pd.Series(ols.predict(X_test), index=test_hours.index, name='price_forecast')

print(f'Hourly forecasts: {len(hourly_forecast):,}')
print(f'Mean predicted price: {hourly_forecast.mean():.2f} EUR/MWh')
print(f'Range: [{hourly_forecast.min():.2f}, {hourly_forecast.max():.2f}] EUR/MWh')

---
## 6 · Aggregate to Weekly Averages

In [ ]:
# ── Assign consecutive week numbers (week 1 = Jan 1–7, ..., week 52 = Dec 23–31) ─
forecast_df = pd.DataFrame({
    'date': test_hours['date'].values,
    'hour': test_hours['hour'].values,
    'price_forecast': hourly_forecast.values,
    'price_actual': test_raw['day_ahead_price_eur_mwh'].values,
}, index=test_hours.index)

day_of_year = forecast_df['date'].dt.dayofyear
forecast_df['week'] = ((day_of_year - 1) // 7 + 1).clip(upper=52)  # cap overflow to week 52

print(f'Hours in 2024: {len(forecast_df):,}')
print(f'Weeks: {forecast_df["week"].min()} to {forecast_df["week"].max()}')
print(f'Days in week 52: {forecast_df.loc[forecast_df["week"]==52, "date"].nunique()}')

# ── Weekly averages: mean price per (week, hour) ─────────────────────────────
weekly_forecast = forecast_df.groupby(['week', 'hour'])['price_forecast'].mean().unstack('hour')
weekly_actual = forecast_df.groupby(['week', 'hour'])['price_actual'].mean().unstack('hour')

weekly_forecast.columns = list(range(24))
weekly_actual.columns = list(range(24))

print(f'\nWeekly forecast table: {weekly_forecast.shape}  (weeks × hours)')
print(f'Weekly actual table:   {weekly_actual.shape}')
print(f'Any NaN in forecast: {weekly_forecast.isna().any().any()}')

In [ ]:
# ── Median forecast: median of same-week prices from 2018–2023 ────────────────
# For each (week, hour), compute the median price across all days in that week
# from all training years. Uses the same consecutive-week definition.

train_prices = train[['day_ahead_price_eur_mwh']].copy()
train_prices['hour'] = train['hour']
train_prices['day_of_year'] = train_prices.index.dayofyear
train_prices['week'] = ((train_prices['day_of_year'] - 1) // 7 + 1).clip(upper=52)

weekly_median = (
    train_prices
    .groupby(['week', 'hour'])['day_ahead_price_eur_mwh']
    .median()
    .unstack('hour')
)
weekly_median.columns = list(range(24))

# Ensure all 52 weeks are present (should be, given 6 years of data)
assert weekly_median.shape == (52, 24), f'Unexpected shape: {weekly_median.shape}'

print(f'Median forecast table: {weekly_median.shape}')
print(f'Mean median price: {weekly_median.values.mean():.2f} EUR/MWh')

---
## 7 · Evaluation

In [ ]:
# ── Overall metrics (all 1,248 weekly-hour forecasts) ─────────────────────────
all_pred = weekly_forecast.values.flatten()
all_actual = weekly_actual.values.flatten()
all_median = weekly_median.values.flatten()

overall = evaluate('Calendar OLS', all_actual, all_pred)
overall_median = evaluate('Median', all_actual, all_median)

print('=== Overall Metrics (Task 2: 1,248 weekly-hour forecasts) ===')
comparison = pd.DataFrame([overall, overall_median])
print(comparison)
print()

In [ ]:
# ── Per-week MAE comparison: Calendar OLS vs Median ──────────────────────────
week_mae_rows = []
for w in weekly_forecast.index:
    w_actual = weekly_actual.loc[w].values.astype(float)
    w_ols = weekly_forecast.loc[w].values.astype(float)
    w_med = weekly_median.loc[w].values.astype(float)
    week_mae_rows.append({
        'Week': int(w),
        'OLS MAE': round(mae(w_actual, w_ols), 2),
        'Median MAE': round(mae(w_actual, w_med), 2),
    })

week_mae_df = pd.DataFrame(week_mae_rows).set_index('Week')

# Append overall row
week_mae_df.loc['Overall'] = [
    round(mae(all_actual, all_pred), 2),
    round(mae(all_actual, all_median), 2),
]

print('=== Per-Week MAE: Calendar OLS vs Median ===')
week_mae_df

In [ ]:
# ── Per-hour metrics ──────────────────────────────────────────────────────────
hour_metrics = []
for k in range(24):
    h_pred = weekly_forecast[k].values.astype(float)
    h_actual = weekly_actual[k].values.astype(float)
    hour_metrics.append(evaluate(f'Hour {k:02d}', h_actual, h_pred))

hour_metrics_df = pd.DataFrame(hour_metrics)
print('=== Per-Hour Metrics (averaged over 52 weeks) ===')
hour_metrics_df

---
## 8 · Visualization

In [ ]:
# ── Plot 1: Weekly average price profiles — selected weeks ─────────────────────
sample_weeks = [1, 10, 20, 30, 40, 52]
fig, axes = plt.subplots(3, 2, figsize=(16, 14))
axes = axes.flatten()

for i, w in enumerate(sample_weeks):
    ax = axes[i]
    actual_vals = weekly_actual.loc[w].values.astype(float)
    pred_vals = weekly_forecast.loc[w].values.astype(float)
    median_vals = weekly_median.loc[w].values.astype(float)
    
    ax.plot(range(24), actual_vals, 'k-o', ms=4, lw=1.5, label='Actual')
    ax.plot(range(24), pred_vals, 'r--s', ms=3, lw=1.2, label='Calendar OLS')
    ax.plot(range(24), median_vals, 'b:^', ms=3, lw=1.2, label='Median')
    
    w_mae_ols = mae(actual_vals, pred_vals)
    w_mae_med = mae(actual_vals, median_vals)
    ax.set_title(f'Week {w}  |  OLS MAE={w_mae_ols:.1f}, Median MAE={w_mae_med:.1f}')
    ax.set_xlabel('Hour')
    ax.set_ylabel('EUR/MWh')
    ax.set_xticks(range(0, 24, 2))
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

fig.suptitle('Task 2: Weekly Average Hourly Profiles — Calendar OLS vs Median', fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Error heatmap (52 weeks × 24 hours) ──────────────────────────────
error_table = weekly_forecast.values - weekly_actual.values
error_df = pd.DataFrame(
    error_table,
    index=[f'W{w}' for w in weekly_forecast.index],
    columns=[f'{k:02d}' for k in range(24)],
)

fig, ax = plt.subplots(figsize=(18, 14))
vabs = np.abs(error_table).max()
sns.heatmap(
    error_df.astype(float), ax=ax, cmap='RdBu_r', center=0,
    vmin=-vabs, vmax=vabs,
    linewidths=0.1, cbar_kws={'label': 'Error (EUR/MWh)'},
)
ax.set_xlabel('Hour of day')
ax.set_ylabel('Week')
ax.set_title('Signed Forecast Error: Predicted − Actual (EUR/MWh)')
fig.tight_layout()
plt.show()

In [ ]:
# ── Plot 3: Weekly mean price time series ─────────────────────────────────────
weekly_mean_actual = weekly_actual.mean(axis=1)
weekly_mean_pred = weekly_forecast.mean(axis=1)
weekly_mean_median = weekly_median.mean(axis=1)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(weekly_mean_actual.index, weekly_mean_actual.values, 'k-o', ms=3, lw=1.5, label='Actual')
ax.plot(weekly_mean_pred.index, weekly_mean_pred.values, 'r--s', ms=3, lw=1.2, label='Calendar OLS')
ax.plot(weekly_mean_median.index, weekly_mean_median.values, 'b:^', ms=3, lw=1.2, label='Median')
ax.set_xlabel('Week')
ax.set_ylabel('Mean Price (EUR/MWh)')
ax.set_title('Weekly Mean Price: Actual vs Forecasts')
ax.legend()
sns.despine(ax=ax)
fig.tight_layout()
plt.show()

---
## 9 · Export

In [ ]:
# ── Export hourly forecasts ───────────────────────────────────────────────────
hourly_export = forecast_df[['date', 'hour', 'price_forecast', 'price_actual']].copy()
hourly_export.to_csv(Path('..') / 'analyses' / 'calendar_hourly_forecast_task2.csv', index=False)
print('Saved: analyses/calendar_hourly_forecast_task2.csv')

# ── Export weekly forecasts (Calendar OLS) ────────────────────────────────────
weekly_export = weekly_forecast.copy()
weekly_export.columns = [f'hour_{k:02d}' for k in range(24)]
weekly_export.index.name = 'week'
weekly_export.to_csv(Path('..') / 'analyses' / 'calendar_forecast_task2.csv')
print('Saved: analyses/calendar_forecast_task2.csv')

# ── Export weekly forecasts (Median) ──────────────────────────────────────────
median_export = weekly_median.copy()
median_export.columns = [f'hour_{k:02d}' for k in range(24)]
median_export.index.name = 'week'
median_export.to_csv(Path('..') / 'analyses' / 'median_forecast_task2.csv')
print('Saved: analyses/median_forecast_task2.csv')

# ── Export actual weekly prices ───────────────────────────────────────────────
actual_export = weekly_actual.copy()
actual_export.columns = [f'hour_{k:02d}' for k in range(24)]
actual_export.index.name = 'week'
actual_export.to_csv(Path('..') / 'analyses' / 'actual_prices_task2.csv')
print('Saved: analyses/actual_prices_task2.csv')
print()

# ── Final summary ────────────────────────────────────────────────────────────
print('=' * 55)
print('  TASK 2 — Summary')
print('=' * 55)
print()
print(f'Training period: 2018-01-01 to {TRAIN_END}')
print(f'Forecast period: 2024 (52 weeks)')
print(f'Total forecasts: {weekly_forecast.shape[0]} × {weekly_forecast.shape[1]} = {weekly_forecast.size}')
print(f'Calendar OLS R² (in-sample): {r2:.4f}')
print()
print('Overall comparison:')
print(comparison.to_string())
print()
print('Best 5 weeks by OLS MAE:')
print(week_mae_df.iloc[:-1].nsmallest(5, 'OLS MAE').to_string())
print()
print('Worst 5 weeks by OLS MAE:')
print(week_mae_df.iloc[:-1].nlargest(5, 'OLS MAE').to_string())